# 🧠 CPT Navigation Trainer (v2)
Entrenamiento de intuición de movimiento.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import io

# 1. Datos Incrustados
csv_data = """f0,f1,f2,f3,f4,f5,f6,f7,success
0.5000,0.5000,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5625,0.5000,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5625,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.5625,0.5833,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5625,0.5833,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.6250,0.5833,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6250,0.5833,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6250,0.6667,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6250,0.6667,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.6875,0.6667,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6875,0.6667,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6875,0.7500,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6875,0.7500,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7500,0.7500,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7500,0.7500,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7500,0.8333,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7500,0.8333,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8125,0.8333,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8125,0.8333,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8125,0.9167,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8125,0.9167,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8750,0.9167,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8750,0.9167,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8750,1.0000,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6362,0.6400,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.6987,0.6400,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6987,0.6400,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6987,0.7233,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6987,0.7233,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7612,0.7233,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7612,0.7233,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7612,0.8067,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7612,0.8067,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8237,0.8067,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8237,0.8067,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8237,0.8900,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8237,0.8900,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8862,0.8900,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8862,0.8900,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8862,0.9733,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4813,0.4550,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5437,0.4550,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5437,0.4550,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.5437,0.5383,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5437,0.5383,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.6062,0.5383,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6062,0.5383,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6062,0.6217,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6062,0.6217,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.6687,0.6217,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6687,0.6217,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6687,0.7050,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6687,0.7050,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7312,0.7050,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7312,0.7050,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7312,0.7883,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7312,0.7883,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7937,0.7883,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7937,0.7883,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7937,0.8717,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7937,0.8717,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8562,0.8717,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8562,0.8717,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8562,0.9550,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6062,0.7433,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.6687,0.7433,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6687,0.7433,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6687,0.8267,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6687,0.8267,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7312,0.8267,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7312,0.8267,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7312,0.9100,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7312,0.9100,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7937,0.9100,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7937,0.9100,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7937,0.9933,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6775,0.2267,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7400,0.2267,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7400,0.2267,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.7400,0.3100,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7400,0.3100,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8025,0.3100,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8025,0.3100,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.8025,0.3933,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8025,0.3933,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8650,0.3933,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8650,0.3933,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.8650,0.4767,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8650,0.4767,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9275,0.4767,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9275,0.4767,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9275,0.5600,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.9275,0.5600,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9900,0.5600,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9900,0.5600,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9900,0.6433,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.3075,0.8183,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.3700,0.8183,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.3700,0.8183,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.3700,0.9017,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6725,0.3117,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7350,0.3117,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7350,0.3117,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.7350,0.3950,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7350,0.3950,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7975,0.3950,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7975,0.3950,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.7975,0.4783,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7975,0.4783,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8600,0.4783,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8600,0.4783,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8600,0.5617,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8600,0.5617,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9225,0.5617,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9225,0.5617,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9225,0.6450,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.9225,0.6450,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9850,0.6450,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9850,0.6450,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9850,0.7283,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5625,0.4267,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.6250,0.4267,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6250,0.4267,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.6250,0.5100,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6250,0.5100,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.6875,0.5100,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6875,0.5100,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6875,0.5933,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6875,0.5933,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7500,0.5933,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7500,0.5933,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7500,0.6767,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7500,0.6767,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8125,0.6767,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8125,0.6767,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8125,0.7600,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8125,0.7600,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8750,0.7600,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8750,0.7600,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8750,0.8433,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8750,0.8433,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9375,0.8433,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9375,0.8433,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9375,0.9267,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4425,0.3083,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5050,0.3083,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5050,0.3083,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.5050,0.3917,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5050,0.3917,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5675,0.3917,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5675,0.3917,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.5675,0.4750,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5675,0.4750,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.6300,0.4750,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6300,0.4750,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6300,0.5583,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6300,0.5583,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.6925,0.5583,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6925,0.5583,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6925,0.6417,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6925,0.6417,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7550,0.6417,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7550,0.6417,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7550,0.7250,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7550,0.7250,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8175,0.7250,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8175,0.7250,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8175,0.8083,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8175,0.8083,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8800,0.8083,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8800,0.8083,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8800,0.8917,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8800,0.8917,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9425,0.8917,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9425,0.8917,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9425,0.9750,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6062,0.6350,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.6687,0.6350,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6687,0.6350,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6687,0.7183,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6687,0.7183,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7312,0.7183,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7312,0.7183,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7312,0.8017,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7312,0.8017,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7937,0.8017,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7937,0.8017,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7937,0.8850,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7937,0.8850,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8562,0.8850,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8562,0.8850,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8562,0.9683,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4375,0.4233,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5000,0.4233,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5000,0.4233,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.5000,0.5067,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5000,0.5067,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5625,0.5067,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5625,0.5067,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.5625,0.5900,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7812,0.4183,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8438,0.4183,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8438,0.4183,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.8438,0.5017,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8438,0.5017,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9062,0.5017,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9062,0.5017,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9062,0.5850,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.9062,0.5850,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9688,0.5850,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9688,0.5850,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9688,0.6683,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6963,0.3517,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7588,0.3517,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7588,0.3517,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.7588,0.4350,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7588,0.4350,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8213,0.4350,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8213,0.4350,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.8213,0.5183,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8213,0.5183,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8838,0.5183,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8838,0.5183,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8838,0.6017,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8838,0.6017,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9463,0.6017,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9463,0.6017,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9463,0.6850,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.3613,0.3900,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.4238,0.3900,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.4238,0.3900,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.4238,0.4733,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4238,0.4733,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.4863,0.4733,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.4863,0.4733,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.4863,0.5567,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4863,0.5567,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5487,0.5567,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5487,0.5567,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.5487,0.6400,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5487,0.6400,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.6112,0.6400,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6112,0.6400,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6112,0.7233,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6112,0.7233,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.6737,0.7233,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6737,0.7233,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6737,0.8067,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6737,0.8067,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7362,0.8067,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7362,0.8067,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7362,0.8900,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7362,0.8900,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7987,0.8900,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7987,0.8900,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7987,0.9733,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.2675,0.3517,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.3300,0.3517,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.3300,0.3517,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.3300,0.4350,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.3300,0.4350,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.3925,0.4350,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.3925,0.4350,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.3925,0.5183,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.3925,0.5183,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.4550,0.5183,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.4550,0.5183,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.4550,0.6017,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4550,0.6017,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5175,0.6017,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5175,0.6017,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.5175,0.6850,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5175,0.6850,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5800,0.6850,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5800,0.6850,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.5800,0.7683,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5800,0.7683,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.6425,0.7683,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6425,0.7683,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6425,0.8517,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6425,0.8517,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7050,0.8517,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7050,0.8517,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7050,0.9350,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7788,0.4133,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8413,0.4133,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8413,0.4133,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.8413,0.4967,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8413,0.4967,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9038,0.4967,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9038,0.4967,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9038,0.5800,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.9038,0.5800,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9663,0.5800,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9663,0.5800,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9663,0.6633,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.3400,0.2517,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.4025,0.2517,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.4025,0.2517,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.4025,0.3350,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6438,0.6383,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7063,0.6383,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7063,0.6383,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7063,0.7217,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7063,0.7217,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7688,0.7217,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7688,0.7217,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7688,0.8050,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7688,0.8050,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8313,0.8050,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8313,0.8050,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8313,0.8883,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8313,0.8883,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8938,0.8883,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8938,0.8883,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8938,0.9717,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.2637,0.5633,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.3262,0.5633,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.3262,0.5633,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.3262,0.6467,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.3262,0.6467,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.3887,0.6467,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.3887,0.6467,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.3887,0.7300,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.3887,0.7300,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.4512,0.7300,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.4512,0.7300,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.4512,0.8133,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4512,0.8133,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5138,0.8133,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5138,0.8133,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.5138,0.8967,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5138,0.8967,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5763,0.8967,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5763,0.8967,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.5763,0.9800,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8488,0.5717,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9113,0.5717,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9113,0.5717,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9113,0.6550,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.9113,0.6550,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9738,0.6550,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9738,0.6550,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9738,0.7383,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4562,0.3500,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5188,0.3500,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5188,0.3500,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.5188,0.4333,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5188,0.4333,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5813,0.4333,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5813,0.4333,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.5813,0.5167,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5813,0.5167,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.6438,0.5167,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6438,0.5167,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6438,0.6000,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6438,0.6000,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7063,0.6000,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7063,0.6000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7063,0.6833,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7063,0.6833,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7688,0.6833,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7688,0.6833,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7688,0.7667,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7688,0.7667,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8313,0.7667,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8313,0.7667,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8313,0.8500,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8313,0.8500,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8938,0.8500,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8938,0.8500,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8938,0.9333,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8375,0.6483,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9000,0.6483,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9000,0.6483,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9000,0.7317,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.9000,0.7317,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9625,0.7317,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9625,0.7317,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9625,0.8150,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.3575,0.5267,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.4200,0.5267,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.4200,0.5267,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.4200,0.6100,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4200,0.6100,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.4825,0.6100,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.4825,0.6100,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.4825,0.6933,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4825,0.6933,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5450,0.6933,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5450,0.6933,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.5450,0.7767,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5450,0.7767,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.6075,0.7767,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6075,0.7767,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6075,0.8600,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6075,0.8600,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.6700,0.8600,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6700,0.8600,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.6700,0.9433,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.1275,0.3967,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.1900,0.3967,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.1900,0.3967,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.1900,0.4800,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.1900,0.4800,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.2525,0.4800,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.2525,0.4800,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.2525,0.5633,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.2525,0.5633,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.3150,0.5633,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.3150,0.5633,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.3150,0.6467,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.3150,0.6467,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.3775,0.6467,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.3775,0.6467,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.3775,0.7300,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.3775,0.7300,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.4400,0.7300,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.4400,0.7300,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.4400,0.8133,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4400,0.8133,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5025,0.8133,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5025,0.8133,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.5025,0.8967,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5025,0.8967,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5650,0.8967,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5650,0.8967,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.5650,0.9800,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5962,0.4033,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.6587,0.4033,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6587,0.4033,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.6587,0.4867,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6587,0.4867,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7212,0.4867,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7212,0.4867,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7212,0.5700,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7212,0.5700,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7837,0.5700,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7837,0.5700,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7837,0.6533,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7837,0.6533,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8462,0.6533,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8462,0.6533,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8462,0.7367,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8462,0.7367,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9087,0.7367,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9087,0.7367,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9087,0.8200,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.9087,0.8200,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9712,0.8200,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9712,0.8200,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9712,0.9033,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8137,0.5783,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8762,0.5783,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8762,0.5783,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8762,0.6617,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8762,0.6617,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9387,0.6617,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9387,0.6617,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9387,0.7450,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4675,0.2567,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5300,0.2567,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5300,0.2567,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.5300,0.3400,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5300,0.3400,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5925,0.3400,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5925,0.3400,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.5925,0.4233,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.5925,0.4233,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.6550,0.4233,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.6550,0.4233,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.6550,0.5067,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.6550,0.5067,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7175,0.5067,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7175,0.5067,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7175,0.5900,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7175,0.5900,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.7800,0.5900,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.7800,0.5900,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.7800,0.6733,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.7800,0.6733,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.8425,0.6733,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.8425,0.6733,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.8425,0.7567,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.8425,0.7567,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9050,0.7567,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9050,0.7567,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9050,0.8400,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.9050,0.8400,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,0
0.9675,0.8400,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.9675,0.8400,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,0
0.9675,0.9233,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4350,0.2950,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.4975,0.2950,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.4975,0.2950,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.4975,0.3783,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
0.4975,0.3783,0.0000,0.0000,0.0000,0.0000,0.0500,0.0000,1
0.5600,0.3783,0.0500,0.0000,0.0000,0.0000,-0.0500,0.0000,0
0.5600,0.3783,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500,1
0.5600,0.4617,0.0000,0.0500,0.0000,0.0000,0.0000,-0.0500,0
"""
df = pd.read_csv(io.StringIO(csv_data))
print(f'Cargadas {len(df)} muestras')

In [ ]:
# 2. Definir Red Estándar CPT
class TabularNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = TabularNet()
opt = optim.Adam(model.parameters(), lr=0.005)
loss_fn = nn.BCELoss()

In [ ]:
# 3. Entrenamiento
X = torch.tensor(df.drop('success', axis=1).values, dtype=torch.float32)
y = torch.tensor(df['success'].values, dtype=torch.float32).view(-1, 1)

for epoch in range(200):
    opt.zero_grad()
    outputs = model(X)
    loss = loss_fn(outputs, y)
    loss.backward()
    opt.step()
    if epoch % 50 == 0: print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

torch.save(model.state_dict(), 'action_tabular_filter.pt')
print('✅ Modelo de Navegación Guardado')

In [ ]:
# 4. Descargar / Guardar
import shutil, os
output_file = 'action_tabular_filter.pt'
if os.path.exists('/kaggle/working'):
    print('Model deployed.')
    print('✅ Modelo guardado en Kaggle Output')
else:
    try:
        from google.colab import files
        files.download(output_file)
        print('✅ Modelo descargado desde Colab')
    except ImportError:
        print('✅ Modelo guardado localmente')
